# NB2 · Veri: MIMIC-IV demo verisinin ağdan çağrılması
### Data: calling the MIMIC-IV demo over the web

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr  
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

---


## Bu defter ne yapıyor

MIMIC-IV Clinical Database Demo, yüz hastalık açık erişimli bir yoğun bakım veri
kümesidir. Açık Veri Ortak Alanı Açık Veritabanı Lisansı ile yayımlanmaktadır, kimlik
doğrulaması gerektirmemektedir ve dosyaları doğrudan HTTPS üzerinden okunabilmektedir.
Bu defter tam olarak bunu yapmaktadır: Veri indirilip yüklenmez, çağrılır.

> Johnson, A., Bulgarelli, L., Pollard, T., Horng, S., Celi, L. A., & Mark, R. (2023).
> MIMIC-IV Clinical Database Demo (version 2.2). PhysioNet.
> [doi:10.13026/dp1f-ex47](https://doi.org/10.13026/dp1f-ex47)

### Ortak senaryo

| | |
|---|---|
| **Hedef** | Uzamış yoğun bakım kalışı, yani üç günden uzun yatış |
| **Karar anı** | Yoğun bakıma kabulden altı saat sonra |
| **Kullanıcı** | Kapasite ve yükseltme planlaması yapan yoğun bakım hekimi |
| **Girdiler** | Demografik bilgi, yatış bağlamı, ilk altı saatin vital ve laboratuvar değerleri |

### İki şerit

**Birinci şerit:** Hücreleri sırayla çalıştırınız. Python bilmeniz gerekmez.  
**İkinci şerit:** Aynı sonucu istem kütüphanesindeki istemlerle kendiniz ürettiriniz.

Oturum boyunca her iki şerit de kullanılmaktadır. Hazır hücre neyin üretilmesi
gerektiğini gösterir, istem ise onu sizin probleminiz için yeniden üretir.


## 1. Bağlantı denetimi · Connection check

Arşivdeki en küçük dosya çekilerek ağ yolunun çalıştığı doğrulanmaktadır. Mekân bu
sunucuyu engelliyorsa bu hücre bir saniye içinde hata verir ve sentetik veri şeridine
geçmek için vakit kalır.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/mimic_web.py', 'mimic_web.py')

import mimic_web as mw

mw.check_connection()


## 2. Kohortun kurulması · Building the cohort

Yedi tablo çağrılmakta ve her yoğun bakım yatışı için tek satırlık bir kayıt
oluşturulmaktadır. Modüldeki iki koruma önemlidir ve kod okunmadan da bilinmelidir.

Birincisi **zaman penceresi**: Her ölçüm yalnızca yatıştan sonraki ilk altı saat
içindeyse alınmaktadır. Sonraki ölçümler karar anında mevcut olmadığı için düşmektedir.

İkincisi **sızıntı koruması**: `los`, `outtime` ve `last_careunit` sütunları
silinmektedir. Bunlar yatış bittikten sonra bilinen değerlerdir ve modele girerlerse
mükemmel bir başarım üretip hiçbir şey öğretmezler.


In [ ]:
cohort = mw.build_cohort()


In [ ]:
cohort.head()


## 3. Ne görüyoruz · What the summary says

Yukarıdaki özet iyi haber vermemektedir ve vermemesi gerekmektedir. Yüz hastalık bir
kümede pozitif vaka sayısı, herhangi bir başarım tahmininin güven aralığını şansı
içerecek kadar geniş bırakmaktadır.

Bu, materyalin kusuru değil dersin kendisidir. 16 Eylül oturumunda anlatılan Epic
Sepsis Model vakası, 38.455 yatış üzerinde dış doğrulama yapıldığında 0,63 eğri altı
alan vermişti. Burada elimizdeki veri onun binde birinden küçüktür. Dördüncü istemin
sonundaki *DEVREYE ALIR MIYDIM* paragrafının cevabı bu senaryoda hayırdır ve bunu
kendi gözünüzle görmeniz beklenmektedir.


In [ ]:
import pandas as pd

features = mw.feature_columns(cohort)
print(f'Modele verilebilecek sütun sayısı: {len(features)}')
print()
print('Eksiklik oranı en yüksek on sütun:')
print(cohort[features].isna().mean().sort_values(ascending=False).head(10).round(2))


## 4. Prevalans egzersizi · The prevalence exercise

Bu hücre, dersteki prevalans eğrisini yeniden üretmektedir. Duyarlılık ve özgüllük
sabittir; değişen tek şey hastalığın kliniğinizdeki sıklığıdır.

Wong ve arkadaşlarının (2021) raporladığı duyarlılık 0,33 ve özgüllük 0,83 değerleri
yüzde 7 prevalansta yüzde 12,7 pozitif kestirim değeri vermektedir; bu, makalede
raporlanan yüzde 12 değerini yeniden üretmektedir.


In [ ]:
mw.prevalence_table(sensitivity=0.33, specificity=0.83).round(3)


In [ ]:
# Kendi rakamlarınızla deneyiniz.
# Hedeflediğiniz durumun prevalansı ile modelinizin çalışma noktasını giriniz.

mw.alerts_per_hundred(sensitivity=0.80, specificity=0.90, prevalence=0.03)


Yukarıdaki çağrıda duyarlılık yüzde 80 ve özgüllük yüzde 90, bir makalede görülse
güçlü bir model sayılırdı. Yüzde 3 prevalansta ise ürettiği uyarıların büyük çoğunluğu
yanlıştır. Kendi hedef durumunuzun prevalansını girerek bu tabloyu kendi kliniğiniz
için okuyunuz.


## 5. İstemlerle devam · Continuing with the prompts

Buradan sonrası istem kütüphanesindedir. Üçüncü ve dördüncü istemleri bu kohort
üzerinde çalıştırınız. Veri çerçevesinin adı `cohort`, sonuç sütunu `prolonged_stay`,
modele verilebilecek sütunlar ise `mw.feature_columns(cohort)` çağrısıyla alınmaktadır.

Asistanınıza aşağıdaki bağlam bloğunu veriniz; hangi dilde çalışacağınızı seçiniz.
Üçüncü istemi iki dilde de çalıştırıp çıktıları karşılaştırmanız beklenmektedir.

### Türkçe bağlam bloğu

```
Elimde `cohort` adında bir pandas veri çerçevesi var. MIMIC-IV demo veri kümesinden
kuruldu. Her satır bir yoğun bakım yatışını temsil ediyor.

Sonuç sütunu: prolonged_stay (1 = yoğun bakım yatışı üç günden uzun)
Kimlik sütunları: subject_id, hadm_id, stay_id, intime
Öznitelikler: demografik bilgi, yatış bağlamı ve yatıştan sonraki ilk altı saate ait
vital ile laboratuvar değerlerinin min, max, mean ve sayım özetleri.

Karar anı yoğun bakıma kabulden altı saat sonrasıdır. Altı saatten sonra kaydedilen
hiçbir bilgi mevcut değildir.

Kohort yüz hastadan oluşmaktadır. Bu kadar küçük bir kümede elde edilen hiçbir
başarım değerini olduğundan güçlü sunma.
```

### English context block

```
I have a pandas dataframe called `cohort`, built from the MIMIC-IV demo dataset.
Each row is one ICU stay.

Outcome column: prolonged_stay (1 = ICU length of stay longer than three days)
Identifier columns: subject_id, hadm_id, stay_id, intime
Features: demographics, admission context, and min, max, mean and count summaries of
vital signs and laboratory results from the first six hours after ICU admission.

The decision point is six hours after ICU admission. Nothing recorded after that
window is available.

The cohort contains one hundred patients. Do not present any performance figure from
a cohort this small as stronger than it is.
```

---

### Uyarı

Bu defterde üretilen hiçbir model doğrulanmış bir klinik araç değildir. MIMIC-IV demo
verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmemektedir. Buradaki çıktılar öğretim amaçlıdır.
